In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="flan_t5_answer_prefix_yes_no_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

def get_single_token_id(text):
    ids = tokenizer(text, add_special_tokens=False).input_ids
    if len(ids) != 1:
        raise ValueError(f"{text!r} is not a single token: {ids} -> {tokenizer.convert_ids_to_tokens(ids)}")
    return ids[0]

yes_token_id = get_single_token_id("yes")
no_token_id = get_single_token_id("no")

answer_prefix = "Answer:"
answer_prefix_ids = tokenizer(answer_prefix, add_special_tokens=False).input_ids
decoder_start_token_id = model.config.decoder_start_token_id

print("model:", model_name)
print("decoder_start_token_id:", decoder_start_token_id)
print("yes_token_id:", yes_token_id, "token:", tokenizer.convert_ids_to_tokens([yes_token_id]))
print("no_token_id:", no_token_id, "token:", tokenizer.convert_ids_to_tokens([no_token_id]))
print("answer_prefix_ids:", answer_prefix_ids, "tokens:", tokenizer.convert_ids_to_tokens(answer_prefix_ids))


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model: google/flan-t5-small
decoder_start_token_id: 0
yes_token_id: 4273 token: ['▁yes']
no_token_id: 150 token: ['▁no']
answer_prefix_ids: [11801, 10] tokens: ['▁Answer', ':']


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

prompts = [
    f"Do these two sentences have the same meaning? Sentence 1: {s1} Sentence 2: {s2}"
    for s1, s2 in zip(sent1, sent2)
]

print("num_examples:", len(ds))
print("positive_rate:", float(y_true.mean()))
print("sample_prompt:", prompts[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
sample_prompt: Do these two sentences have the same meaning? Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy . Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [4]:
batch_size = 16
preds = []
yes_logits_all = []
no_logits_all = []

decoder_prefix_ids = [decoder_start_token_id] + answer_prefix_ids
decoder_prefix = torch.tensor(decoder_prefix_ids, dtype=torch.long, device=device)

with torch.no_grad():
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        batch_decoder_input_ids = decoder_prefix.unsqueeze(0).repeat(len(batch_prompts), 1)

        logits = model(**enc, decoder_input_ids=batch_decoder_input_ids).logits
        step_logits = logits[:, -1, :]

        batch_yes_logits = step_logits[:, yes_token_id]
        batch_no_logits = step_logits[:, no_token_id]
        batch_preds = (batch_yes_logits > batch_no_logits).long().cpu().numpy()

        preds.extend(batch_preds.tolist())
        yes_logits_all.extend(batch_yes_logits.detach().cpu().numpy().tolist())
        no_logits_all.extend(batch_no_logits.detach().cpu().numpy().tolist())

y_pred = np.array(preds)
yes_logits_all = np.array(yes_logits_all)
no_logits_all = np.array(no_logits_all)

print("done")
print("num_predicted_positive:", int(y_pred.sum()))


  0%|          | 0/26 [00:00<?, ?it/s]

done
num_predicted_positive: 195


In [ ]:

vault.create_record_list("flan_prediction_and_logit_prefix", column_names=["prediction","yes_logits", "no_logits"])

for i in range(len(y_pred)):
    vault.append_record("flan_prediction_and_logit_prefix", 
                        {
                            "prediction": y_pred[i],
                            "yes_logits": float(yes_logits_all[i]),
                            "no_logits": float(no_logits_all[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT flan_prediction_and_logit_prefix"
embedding = get_embeddings(description)
vault.create_description("flan_prediction_and_logit_prefix", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_prediction_and_logit_prefix", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], zero_division=0)
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], zero_division=0))


{'accuracy': 0.6372549019607843, 'f1': 0.6877637130801688}
                precision    recall  f1-score   support

not_paraphrase       0.46      0.75      0.57       129
    paraphrase       0.84      0.58      0.69       279

      accuracy                           0.64       408
     macro avg       0.65      0.67      0.63       408
  weighted avg       0.72      0.64      0.65       408



In [6]:
for i in range(5):
    print("=" * 100)
    print("idx:", i)
    print("prompt:", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("yes_logit:", float(yes_logits_all[i]), "no_logit:", float(no_logits_all[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 100)
    print("idx:", int(i))
    print("prompt:", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("yes_logit:", float(yes_logits_all[i]), "no_logit:", float(no_logits_all[i]))


idx: 0
prompt: Do these two sentences have the same meaning? Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy . Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1
yes_logit: -1.049403429031372 no_logit: -1.3706063032150269
idx: 1
prompt: Do these two sentences have the same meaning? Sentence 1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war . Sentence 2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0
yes_logit: -3.331972122192383 no_logit: -1.0732812881469727
idx: 2
prompt: Do these two sentences have the same meaning? Sentence 1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat . Sentence 2: The dollar was at 116.78 yen JPY = , virtually flat on the session , 

In [7]:
vault.create_record_list("flan_t5_answer_prefix_yes_no_mrpcc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("flan_t5_answer_prefix_yes_no_mrpcc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "flan_prediction_and_logit_prefix": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT flan_t5_answer_prefix_yes_no_mrpcc_summary"
embedding = get_embeddings(description)
vault.create_description("flan_t5_answer_prefix_yes_no_mrpcc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_answer_prefix_yes_no_mrpcc_summary", cat, embedding, prop)




{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'google/flan-t5-small',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6372549019607843,
 'f1': 0.6877637130801688}

In [ ]:
description = "INSERT TEXT HERE ABOUT flan_t5_answer_prefix_yes_no_mrpc.ipynb process/notebook" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("flan_t5_answer_prefix_yes_no_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_answer_prefix_yes_no_mrpc", cat, embedding, prop)